# Pyxis Model Analysis — Google Colab (T4)

Three inference experiments on all 3 best models:
1. **Confidence threshold sweep** — find optimal conf for each model
2. **Cross-dataset evaluation** — test each model on every test set
3. **Test-time augmentation (TTA)** — does `augment=True` help?

## Folder structure
```
_Google_finetuning/
├── gold_test/
│   ├── internet_scraped/  (195 images + labels)
│   └── singapore_river/   (39 images + labels)
├── weights/
│   ├── sg_y11s_t0_960.pt
│   ├── is_v2_y11s_960.pt
│   └── combined_y11s_1280.pt
└── Pyxis_Analysis.ipynb
```

## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH
WORK_DIR = '/content/drive/MyDrive/_Google_finetuning'
%cd {WORK_DIR}

Mounted at /content/drive
[Errno 2] No such file or directory: '/content/drive/MyDrive/_Google_finetuning'
/content


In [ ]:
!pip install -q ultralytics>=8.3.0

In [ ]:
import torch
import numpy as np
import csv
from pathlib import Path
from ultralytics import YOLO

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# Paths
WORK = Path(WORK_DIR)
SG_IMAGES = WORK / 'gold_test' / 'singapore_river' / 'images'
SG_LABELS = WORK / 'gold_test' / 'singapore_river' / 'labels'
IS_IMAGES = WORK / 'gold_test' / 'internet_scraped' / 'images'
IS_LABELS = WORK / 'gold_test' / 'internet_scraped' / 'labels'

SG_MODEL = str(WORK / 'weights' / 'sg_y11s_t0_960.pt')
IS_MODEL = str(WORK / 'weights' / 'is_v2_y11s_960.pt')
COMBINED_MODEL = str(WORK / 'weights' / 'combined_y11s_1280.pt')

MODELS = {
    'SG': {'path': SG_MODEL, 'imgsz': 960},
    'IS': {'path': IS_MODEL, 'imgsz': 960},
    'Combined': {'path': COMBINED_MODEL, 'imgsz': 1280},
}

TEST_SETS = {
    'SG test': {'images': SG_IMAGES, 'labels': SG_LABELS},
    'IS test': {'images': IS_IMAGES, 'labels': IS_LABELS},
}

print(f'SG test: {len(list(SG_IMAGES.iterdir()))} images')
print(f'IS test: {len(list(IS_IMAGES.iterdir()))} images')

In [ ]:
# ============================================
# Shared evaluation functions
# ============================================

def load_labels(path, cls_id):
    boxes = []
    if not path.exists():
        return boxes
    for line in path.read_text(encoding='utf-8').strip().split('\n'):
        parts = line.strip().split()
        if len(parts) != 5: continue
        if int(parts[0]) != cls_id: continue
        cx, cy, w, h = map(float, parts[1:5])
        x0, y0 = max(0, cx-w/2), max(0, cy-h/2)
        x1, y1 = min(1, cx+w/2), min(1, cy+h/2)
        boxes.append([x0, y0, x1, y1])
    return boxes

def iou(a, b):
    ix0, iy0 = max(a[0],b[0]), max(a[1],b[1])
    ix1, iy1 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,ix1-ix0)*max(0,iy1-iy0)
    aa = max(0,a[2]-a[0])*max(0,a[3]-a[1])
    ab = max(0,b[2]-b[0])*max(0,b[3]-b[1])
    d = aa+ab-inter
    return inter/d if d>0 else 0.0

def greedy_match(gt, pred, thresh=0.5):
    matched = set()
    tp = 0
    for p in pred:
        best_iou, best_j = 0, -1
        for j, g in enumerate(gt):
            if j in matched: continue
            s = iou(g, p)
            if s > best_iou: best_iou, best_j = s, j
        if best_iou >= thresh and best_j >= 0:
            matched.add(best_j)
            tp += 1
    return tp, len(pred)-tp, len(gt)-tp

def evaluate(model_path, images_dir, labels_dir, conf=0.25, imgsz=960, augment=False):
    model = YOLO(model_path)
    m = {0: {'tp':0,'fp':0,'fn':0}, 1: {'tp':0,'fp':0,'fn':0}}
    imgs = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'}])
    
    for img in imgs:
        lbl = labels_dir / (img.stem + '.txt')
        gt = {0: load_labels(lbl, 0), 1: load_labels(lbl, 1)}
        
        pred = model.predict(str(img), imgsz=imgsz, conf=conf, iou=0.7, verbose=False, augment=augment)[0]
        pred_by_cls = {0: [], 1: []}
        
        if pred.boxes is not None and len(pred.boxes) > 0:
            w_img, h_img = pred.orig_shape[1], pred.orig_shape[0]
            xyxy = pred.boxes.xyxy.cpu().numpy()
            cls_arr = pred.boxes.cls.cpu().numpy().astype(int)
            conf_arr = pred.boxes.conf.cpu().numpy()
            for k in np.argsort(-conf_arr):
                c = cls_arr[k]
                if c not in pred_by_cls: continue
                b = [xyxy[k][0]/w_img, xyxy[k][1]/h_img, xyxy[k][2]/w_img, xyxy[k][3]/h_img]
                pred_by_cls[c].append(b)
        
        for c in [0,1]:
            tp,fp,fn = greedy_match(gt[c], pred_by_cls[c])
            m[c]['tp']+=tp; m[c]['fp']+=fp; m[c]['fn']+=fn
    
    results = {}
    for c, name in [(0,'ladder'),(1,'person')]:
        tp,fp,fn = m[c]['tp'],m[c]['fp'],m[c]['fn']
        r = tp/(tp+fn) if (tp+fn)>0 else 0
        p = tp/(tp+fp) if (tp+fp)>0 else 0
        f1 = 2*r*p/(r+p) if (r+p)>0 else 0
        results[f'{name}_recall'] = round(r,4)
        results[f'{name}_precision'] = round(p,4)
        results[f'{name}_f1'] = round(f1,4)
    results['n_images'] = len(imgs)
    return results

print('Evaluation functions loaded.')

---
## Experiment 1: Confidence Threshold Sweep

Find the optimal confidence threshold for each model. Currently using 0.25 — is that the best tradeoff?

In [ ]:
import pandas as pd

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

sweep_data = {}

for model_name, mcfg in MODELS.items():
    # Each model evaluated on BOTH test sets
    for test_name, tcfg in TEST_SETS.items():
        key = f'{model_name} → {test_name}'
        print(f'=== {key} ===')
        rows = []
        for conf in thresholds:
            r = evaluate(mcfg['path'], tcfg['images'], tcfg['labels'], conf=conf, imgsz=mcfg['imgsz'])
            r['conf'] = conf
            rows.append(r)
            print(f"  conf={conf:.2f} | Ladder R={r['ladder_recall']:.3f} P={r['ladder_precision']:.3f} F1={r['ladder_f1']:.3f} | Person R={r['person_recall']:.3f} P={r['person_precision']:.3f}")
        df = pd.DataFrame(rows)
        sweep_data[key] = df
        print(f"  Best ladder F1 at conf={df.loc[df['ladder_f1'].idxmax(), 'conf']}")
        print()

In [ ]:
# Plot threshold sweep — one subplot per model, showing in-domain test set
import matplotlib.pyplot as plt

# Primary plots: each model on its native test set + combined on both
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_configs = [
    (axes[0], 'SG → SG test', 'SG model on SG test'),
    (axes[1], 'IS → IS test', 'IS model on IS test'),
    (axes[2], 'Combined → IS test', 'Combined model on IS test'),
]

for ax, key, title in plot_configs:
    df = sweep_data[key]
    ax.plot(df['conf'], df['ladder_recall'], 'b-o', label='Ladder Recall')
    ax.plot(df['conf'], df['ladder_precision'], 'b--s', label='Ladder Precision')
    ax.plot(df['conf'], df['ladder_f1'], 'b:^', label='Ladder F1', linewidth=2)
    ax.plot(df['conf'], df['person_recall'], 'r-o', label='Person Recall')
    ax.plot(df['conf'], df['person_precision'], 'r--s', label='Person Precision')
    ax.plot(df['conf'], df['person_f1'], 'r:^', label='Person F1', linewidth=2)
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('threshold_sweep.png', dpi=150)
plt.show()
print('Saved: threshold_sweep.png')

---
## Experiment 2: Cross-Dataset Evaluation

How well does each model generalize to the other dataset?

In [ ]:
print('=== CROSS-DATASET EVALUATION (all 3 models × 2 test sets) ===')
print()

cross_results = []

for model_name, mcfg in MODELS.items():
    for test_name, tcfg in TEST_SETS.items():
        r = evaluate(mcfg['path'], tcfg['images'], tcfg['labels'], conf=0.25, imgsz=mcfg['imgsz'])
        r['model'] = f'{model_name} model'
        r['test_set'] = test_name
        cross_results.append(r)
        print(f"{model_name} → {test_name}: Ladder R={r['ladder_recall']:.3f} P={r['ladder_precision']:.3f} F1={r['ladder_f1']:.3f} | Person R={r['person_recall']:.3f} P={r['person_precision']:.3f} F1={r['person_f1']:.3f}")

df_cross = pd.DataFrame(cross_results)
print()
print(df_cross[['model','test_set','ladder_recall','ladder_precision','ladder_f1','person_recall','person_precision','person_f1']].to_string(index=False))

In [ ]:
# Plot cross-dataset as grouped bar chart (3 models × 2 test sets = 6 bars)
import matplotlib.pyplot as plt
import numpy as np

labels = [f"{r['model']}\n→ {r['test_set']}" for r in cross_results]
ladder_r = [r['ladder_recall'] for r in cross_results]
ladder_p = [r['ladder_precision'] for r in cross_results]
person_r = [r['person_recall'] for r in cross_results]
person_p = [r['person_precision'] for r in cross_results]

x = np.arange(len(labels))
w = 0.2

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - 1.5*w, ladder_r, w, label='Ladder Recall', color='#2196F3')
ax.bar(x - 0.5*w, ladder_p, w, label='Ladder Precision', color='#90CAF9')
ax.bar(x + 0.5*w, person_r, w, label='Person Recall', color='#F44336')
ax.bar(x + 1.5*w, person_p, w, label='Person Precision', color='#EF9A9A')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Cross-Dataset Evaluation — 3 Models × 2 Test Sets')
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

for i, (lr, lp, pr, pp) in enumerate(zip(ladder_r, ladder_p, person_r, person_p)):
    for j, v in enumerate([lr, lp, pr, pp]):
        ax.text(i + (j-1.5)*w, v + 0.02, f'{v:.2f}', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('cross_dataset.png', dpi=150)
plt.show()
print('Saved: cross_dataset.png')

---
## Experiment 3: Test-Time Augmentation (TTA)

Does `model.predict(augment=True)` improve recall without retraining?

In [ ]:
print('=== TEST-TIME AUGMENTATION ===')
print()

tta_results = []

# Test each model on its primary test set
tta_configs = [
    ('SG model', SG_MODEL, 'SG test', SG_IMAGES, SG_LABELS, 960),
    ('IS model', IS_MODEL, 'IS test', IS_IMAGES, IS_LABELS, 960),
    ('Combined model', COMBINED_MODEL, 'IS test', IS_IMAGES, IS_LABELS, 1280),
]

for model_name, model_path, test_name, img_dir, lbl_dir, imgsz in tta_configs:
    # Without TTA
    r_normal = evaluate(model_path, img_dir, lbl_dir, conf=0.25, imgsz=imgsz, augment=False)
    r_normal['model'] = model_name
    r_normal['tta'] = 'OFF'
    tta_results.append(r_normal)
    
    # With TTA
    r_tta = evaluate(model_path, img_dir, lbl_dir, conf=0.25, imgsz=imgsz, augment=True)
    r_tta['model'] = model_name
    r_tta['tta'] = 'ON'
    tta_results.append(r_tta)
    
    # Compare
    lr_diff = r_tta['ladder_recall'] - r_normal['ladder_recall']
    pr_diff = r_tta['person_recall'] - r_normal['person_recall']
    print(f"{model_name} on {test_name}:")
    print(f"  Normal: Ladder R={r_normal['ladder_recall']:.3f} P={r_normal['ladder_precision']:.3f} F1={r_normal['ladder_f1']:.3f} | Person R={r_normal['person_recall']:.3f} P={r_normal['person_precision']:.3f}")
    print(f"  TTA:    Ladder R={r_tta['ladder_recall']:.3f} P={r_tta['ladder_precision']:.3f} F1={r_tta['ladder_f1']:.3f} | Person R={r_tta['person_recall']:.3f} P={r_tta['person_precision']:.3f}")
    print(f"  Delta:  Ladder R {lr_diff:+.3f} | Person R {pr_diff:+.3f}")
    print()

---
## Summary

In [ ]:
print('=' * 70)
print('EXPERIMENT SUMMARY')
print('=' * 70)

print('\n1. CONFIDENCE THRESHOLD — Best Ladder F1')
for key, df in sweep_data.items():
    best_idx = df['ladder_f1'].idxmax()
    print(f'   {key}: conf={df.loc[best_idx, "conf"]}  (F1={df.loc[best_idx, "ladder_f1"]:.3f})')

print('\n2. CROSS-DATASET GENERALIZATION')
for _, row in df_cross.iterrows():
    print(f'   {row["model"]} → {row["test_set"]}: Ladder F1={row["ladder_f1"]:.3f}  Person F1={row["person_f1"]:.3f}')

print('\n3. TEST-TIME AUGMENTATION')
for i in range(0, len(tta_results), 2):
    normal = tta_results[i]
    tta = tta_results[i+1]
    l_diff = tta['ladder_recall'] - normal['ladder_recall']
    p_diff = tta['person_recall'] - normal['person_recall']
    verdict = 'HELPS' if l_diff > 0.01 else 'NO EFFECT' if l_diff > -0.01 else 'HURTS'
    print(f'   {normal["model"]}: Ladder R {normal["ladder_recall"]:.3f} → {tta["ladder_recall"]:.3f} ({l_diff:+.3f}), Person R {normal["person_recall"]:.3f} → {tta["person_recall"]:.3f} ({p_diff:+.3f}) — {verdict}')

print('\n' + '=' * 70)